# Whisper Large v3 Batch Transcription
Reads MP3 files from `My Drive/transcription13_03/Audio`, transcribes them in batches using Whisper Large v3, and writes a CSV to `My Drive/transcription13_03/Output`.

In [1]:
# I install all required dependencies for Whisper Large v3 via HuggingFace
!pip install -q transformers accelerate datasets soundfile librosa ffmpeg-python
!apt-get install -y -q ffmpeg 2>/dev/null

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [2]:
# I mount Google Drive so I can access the Audio folder and write to Output
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import torch
import pandas as pd
from pathlib import Path
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor

# I verify GPU is available and print device info for confirmation
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
# I authenticate with HuggingFace using the token stored in Colab secrets
# If the token is stored under a different secret name, update the key below
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab secrets.")
except Exception:
    hf_token = None
    print("No HF token found in secrets. Proceeding without authentication (public model).")

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

HF token loaded from Colab secrets.


In [5]:
# I load Whisper Large v3 in float16 to maximise speed on the L4 GPU
# SDPA attention implementation is used for optimal memory efficiency
MODEL_ID = "openai/whisper-large-v3"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading {MODEL_ID} on {device} with {torch_dtype} precision...")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    token=hf_token,
)
model.to(device)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

# I build the pipeline with chunking enabled for long audio files
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=30,
    stride_length_s=5,
)

print("Model loaded successfully.")

Loading openai/whisper-large-v3 on cuda with torch.float16 precision...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Model loaded successfully.


In [6]:
# I define the updated input/output paths and scan for all MP3 files
AUDIO_DIR  = "/content/drive/MyDrive/Audio Transcription/Audio_last/audio"
OUTPUT_DIR = "/content/drive/MyDrive/Audio Transcription/Output"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "transcriptions.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(".mp3")
])

print(f"Found {len(audio_files)} MP3 files in {AUDIO_DIR}")

# I load the checkpoint CSV if it already exists to determine which IDs are done
if os.path.exists(OUTPUT_CSV):
    checkpoint_df = pd.read_csv(OUTPUT_CSV, dtype=str)
    already_done_ids = set(checkpoint_df["ID"].dropna().tolist())
    print(f"Checkpoint found: {len(already_done_ids)} files already transcribed, resuming from where we left off.")
else:
    checkpoint_df = None
    already_done_ids = set()
    print("No checkpoint found, starting fresh.")

Found 3424 MP3 files in /content/drive/MyDrive/Audio Transcription/Audio_last/audio
No checkpoint found, starting fresh.


In [8]:
import shutil
from tqdm.notebook import tqdm

LOCAL_AUDIO_DIR = "/content/audio_cache"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

print("Copying audio files to local disk...")
for fname in tqdm(audio_files, desc="Copying", unit="file"):
    src = os.path.join(AUDIO_DIR, fname)
    dst = os.path.join(LOCAL_AUDIO_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

print("Copy done, reads will now come from local NVMe.")

# I update AUDIO_DIR to point at the local cache for the rest of the pipeline
AUDIO_DIR = LOCAL_AUDIO_DIR

Copying audio files to local disk...


Copying:   0%|          | 0/3424 [00:00<?, ?file/s]

Copy done, reads will now come from local NVMe.


In [9]:
# I parse the numeric ID from each filename and build a records list,
# skipping any files whose ID is already present in the checkpoint CSV
records = []
skipped = 0

for fname in audio_files:
    file_id = fname.split('_')[0]
    if file_id in already_done_ids:
        skipped += 1
        continue
    records.append({
        "ID": file_id,
        "original_file_name": fname,
        "file_path": os.path.join(AUDIO_DIR, fname),
        "transcribed_text": "",
    })

print(f"Skipped (already done) : {skipped}")
print(f"Pending transcription  : {len(records)}")

if records:
    print("\nSample pending records:")
    for r in records[:3]:
        print(f"  ID={r['ID']}  file={r['original_file_name']}")
else:
    print("Nothing left to process, all files are already transcribed.")

Skipped (already done) : 0
Pending transcription  : 3424

Sample pending records:
  ID=1000090828  file=1000090828_DECK_OF_CREATIONS.mp3
  ID=1000154193  file=1000154193_Euphorium.mp3
  ID=1000216800  file=1000216800_Mothers_Into_Living_Fit_Yoga_DVD_for_Moms_and_Babies.mp3


In [ ]:
# I process pending files in batches and write each completed batch to the
# checkpoint CSV immediately so a runtime restart only loses the current batch
from tqdm.notebook import tqdm

BATCH_SIZE = 16

generate_kwargs = {
    "language": "english",
    "task": "transcribe",
}

file_paths        = [r["file_path"] for r in records]
total_batches     = (len(file_paths) + BATCH_SIZE - 1) // BATCH_SIZE if file_paths else 0
all_transcriptions = []

batch_ranges = list(range(0, len(file_paths), BATCH_SIZE))

with tqdm(total=len(file_paths), desc="Transcribing", unit="file") as pbar:
    for batch_idx in batch_ranges:
        batch_paths   = file_paths[batch_idx : batch_idx + BATCH_SIZE]
        batch_records = records[batch_idx : batch_idx + BATCH_SIZE]
        batch_num     = batch_idx // BATCH_SIZE + 1
        batch_texts   = []

        try:
            results = asr_pipe(
                batch_paths,
                generate_kwargs=generate_kwargs,
                batch_size=BATCH_SIZE,
                return_timestamps=False,
            )

            if isinstance(results, dict):
                results = [results]

            for res in results:
                batch_texts.append(res["text"].strip())

        except Exception as e:
            tqdm.write(f"  ERROR on batch {batch_num}: {e}")
            # I fall back to one-by-one processing when a whole batch fails
            for path in batch_paths:
                try:
                    res = asr_pipe(path, generate_kwargs=generate_kwargs, return_timestamps=False)
                    batch_texts.append(res["text"].strip())
                except Exception as inner_e:
                    tqdm.write(f"    FAILED {os.path.basename(path)}: {inner_e}")
                    batch_texts.append("TRANSCRIPTION_ERROR")

        all_transcriptions.extend(batch_texts)

        # I write this batch to the checkpoint CSV immediately after it completes
        batch_rows = pd.DataFrame([
            {
                "ID": batch_records[i]["ID"],
                "original_file_name": batch_records[i]["original_file_name"],
                "transcribed_text": batch_texts[i],
            }
            for i in range(len(batch_texts))
        ])

        write_header = not os.path.exists(OUTPUT_CSV)
        batch_rows.to_csv(OUTPUT_CSV, mode="a", index=False, encoding="utf-8", header=write_header)

        pbar.update(len(batch_paths))
        pbar.set_postfix({"batch": f"{batch_num}/{total_batches}", "checkpoint": "saved"})

tqdm.write(f"\nDone. {len(all_transcriptions)} new files transcribed this session.")

Transcribing:   0%|          | 0/3424 [00:00<?, ?file/s]

In [ ]:
# I reload the full output CSV which now includes all prior checkpoint data
# plus everything transcribed this session, then display a summary
df = pd.read_csv(OUTPUT_CSV, dtype=str)

print(f"CSV saved to : {OUTPUT_CSV}")
print(f"Total rows   : {len(df)}")
df.head()

CSV saved to: /content/drive/MyDrive/transcription13_03/Output/transcriptions.csv
Total rows: 1567


,ID,original_file_name,transcribed_text
0,1001512038,1001512038_Feel_Good_Mat__A_New_Way_to_Relax__...,We are feel-good people. We believe feeling go...
1,1001767271,1001767271_Terminally_Chill.mp3,"Hi, I'm Kristen. And I'm Matt. And together, w..."
2,1002421792,1002421792__Fragile__Stephanie_Mathias__First_...,I'm Stephanie Mathias. I am a singer-songwrite...
3,100613576,100613576_Funding_a_Veterans_second_chance_by_...,"If you've clicked on this link, you're possibl..."
4,1006230786,1006230786_The_Sleep_Sensei_-_Fall_Asleep_Fast...,"Hi, I'm Jeremy, and I'd like to introduce you ..."


In [ ]:
# I print a quick sanity check to confirm error rate across all transcribed files
error_count = (df["transcribed_text"] == "TRANSCRIPTION_ERROR").sum()
missing_count = (df["transcribed_text"] == "MISSING").sum()
success_count = len(df) - error_count - missing_count

print(f"Successful transcriptions : {success_count}")
print(f"Errors                    : {error_count}")
print(f"Missing                   : {missing_count}")

if error_count > 0 or missing_count > 0:
    print("\nFailed files:")
    failed = df[df["transcribed_text"].isin(["TRANSCRIPTION_ERROR", "MISSING"])]
    print(failed[["ID", "original_file_name"]].to_string(index=False))

Successful transcriptions : 1567
Errors                    : 0
Missing                   : 0
